In [1]:
# Installing necessary libraries
!pip install kagglehub

In [2]:
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-oss-7.9.2-linux-x86_64.tar.gz.sha512
!tar -xzf elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!sudo chown -R daemon:daemon elasticsearch-7.9.2/
!shasum -a 512 -c elasticsearch-oss-7.9.2-linux-x86_64.tar.gz.sha512

elasticsearch-oss-7.9.2-linux-x86_64.tar.gz: OK


In [3]:
# https://stackoverflow.com/questions/68762774/elasticsearchunsupportedproducterror-the-client-noticed-that-the-server-is-no#answer-68918449
!pip install elasticsearch==7.9.1 -q

In [4]:
# check elasticsearch version in environment
!pip freeze | grep elasticsearch

elasticsearch==7.9.1


In [5]:
!pip install --upgrade numpy==1.24.3

In [6]:
%%bash --bg
sudo -H -u daemon elasticsearch-7.9.2/bin/elasticsearch

In [7]:
%%bash
ps -ef | grep elasticsearch

root        1602    1600  0 20:46 ?        00:00:00 sudo -H -u daemon elasticsearch-7.9.2/bin/elasticsearch
root        1605    1603  0 20:46 ?        00:00:00 grep elasticsearch


In [8]:
#Importing necessary packages
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm
import kagglehub
from elasticsearch import Elasticsearch
from elasticsearch.exceptions import ElasticsearchException
from elasticsearch.helpers import bulk
import time
import pandas as pd
import spacy
import string
from sklearn.feature_extraction.text import CountVectorizer
import os
import numpy as np

In [9]:
es = Elasticsearch("http://localhost:9200")
# Let's test whether we have succesfully started an ES instance and
# imported the python library
if es.ping():
  print('ES instance working')
else:
  print('ES instance not working')

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.11/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/elasticsearch/connection/http_urllib3.py", line 245, in perform_request
    response = self.pool.urlopen(
               ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py", line 841, in urlopen
    retries = retries.increment(
              ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/pyth

ES instance not working


In [10]:
# start es server
time.sleep(20) # give the server 20 seconds to start
!curl -X GET "http://localhost:9200"

{
  "name" : "652e07d77669",
  "cluster_name" : "elasticsearch",
  "cluster_uuid" : "e2IDHoStSQez_6MII4r_ow",
  "version" : {
    "number" : "7.9.2",
    "build_flavor" : "oss",
    "build_type" : "tar",
    "build_hash" : "d34da0ea4a966c4e49417f2da2f244e3e97b4e6e",
    "build_date" : "2020-09-23T00:45:33.626720Z",
    "build_snapshot" : false,
    "lucene_version" : "8.6.2",
    "minimum_wire_compatibility_version" : "6.8.0",
    "minimum_index_compatibility_version" : "6.0.0-beta1"
  },
  "tagline" : "You Know, for Search"
}


In [11]:
# Get, load and visualise Dataset
path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
print("Path to dataset files:", path)

100%|██████████| 242M/242M [00:01<00:00, 167MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/snap/amazon-fine-food-reviews/versions/2


In [12]:
# Construct the full path to Reviews.csv
csv_path = os.path.join(path, "Reviews.csv")

# Load the CSV
reviews = pd.read_csv(csv_path)

In [13]:
# Remove columns not needed
reviews = reviews.drop(['Id', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator','Time'], axis=1)
reviews = reviews.dropna(subset=['Summary'])
reviews.head()

,ProductId,Score,Summary,Text
0,B001E4KFG0,5,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,B00813GRG4,1,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,B000LQOCH0,4,"""Delight"" says it all",This is a confection that has been around a fe...
3,B000UA0QIQ,2,Cough Medicine,If you are looking for the secret ingredient i...
4,B006K2ZZ7K,5,Great taffy,Great taffy at a great price. There was a wid...


In [14]:
## PREPROCESSING MODULE

#Drop Duplicates
reviews = reviews.drop_duplicates(subset=['Text'])

#convert dataframe into a list
reviews_list = reviews.values.tolist()

#extract only the review text column as a list for preprocessing
reviews_text = [row[3] for row in reviews.values.tolist()]

#NLTK resources
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

#tools
wnl = WordNetLemmatizer() #for lemmatisation
regex_punct = r'[^\w\s]'  #for removing punctuation
vectorizer = CountVectorizer(stop_words='english') #for removing stopwords
stopword_set = vectorizer.get_stop_words()

reviews_list = [
    [
        row[0],  # product ID
        row[1],  # score
        row[2],  # summary
        [wnl.lemmatize(t.lower()) for t in word_tokenize(row[3])
         if not re.search(regex_punct, t) and t.lower() not in stopword_set] #text
    ]
    for row in tqdm(reviews_list) if isinstance(row[3], str)
]

#rejoin tokens in row[3] back into strings for index
for row in reviews_list:
    row[3] = ' '.join(row[3])

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
100%|██████████| 393576/393576 [06:20<00:00, 1035.01it/s]


In [15]:
## INDEXING MODULE

# Define the request body for creating an Elasticsearch index
request_body = {
    'settings': {
        'number_of_shards': 1,  # Set the number of primary shards for the index
        'number_of_replicas': 1,  # Set the number of replica shards for the index
        'index': {
            'refresh_interval': '-1'  # Disable automatic refresh to optimize bulk indexing
        },
        'similarity': {
            'default': {
                'type': 'BM25',  # Use BM25 similarity algorithm for ranking search results
                "b": 0.75,  # Controls document field-length normalization, higher value gives more weight to short documents
                "k1": 1.2  # Controls term frequency saturation, higher value gives more weight to frequent terms
            }
        }
    },
    'mappings': {
        'properties': {
            'productid': {'type': 'keyword'},  # 'productid' is a keyword field (not analyzed, exact match)
            'summary': {
                'type': 'text',  # 'summary' is a text field, analyzed for full-text search
                'fields': {
                    'keyword': {'type': 'keyword'}  # Create a 'keyword' sub-field for exact matching on 'summary'
                }
            },
            'text': {
                'type': 'text',  # 'text' field is also analyzed for full-text search
                'fields': {
                    'keyword': {'type': 'keyword'}  # Create a 'keyword' sub-field for exact matching on 'text'
                }
            },
            'rating': {'type': 'integer'}  # 'rating' is an integer field (e.g., product ratings)
        }
    }
}

# Define the index name to use in Elasticsearch
index_name = 'food-reviews'

# Delete the index if it exists
if es.indices.exists(index=index_name):
    print(f"Deleting existing index: {index_name}")
    es.indices.delete(index=index_name)

# Try to check if the index already exists
try:
    es.indices.get(index_name)  # Try to fetch the index details
    print(f'Index {index_name} already exists')  # If it exists, print this message
except:
    # If index doesn't exist, create the index with the provided settings and mappings
    print(f'Creating index {index_name}')
    es.indices.create(index_name, body=request_body)  # Create the index with the defined request body

# Function to index data
def gendata():
    for reviews in reviews_list:
        yield {
            '_op_type': 'index',
            '_index': index_name,
            'productid': reviews[0],
            'summary': reviews[2],
            'text': reviews[3],
            'rating': reviews[1],
        }

# Perform the bulk operation
success = bulk(es, gendata())
print(f"Successfully indexed {success} documents.")
es.indices.refresh(index=index_name)

Creating index food-reviews
Successfully indexed (393576, []) documents.


{'_shards': {'total': 2, 'successful': 1, 'failed': 0}}

In [16]:
##QUERY PROCESSING MODULE

# Function that outputs the documents relevant to the query
def pretty_response(response):
    previous_text = ""
    if len(response["hits"]["hits"]) == 0:
        print("Your search returned no results.")
    else:
        for hit in response["hits"]["hits"]:
            # Access the fields from '_source'
            text = hit["_source"]['text']
            id = hit["_source"]['productid']
            summary = hit["_source"]['summary']
            rating = hit["_source"]["rating"]
            score = hit["_score"]
            explanation = hit["_explanation"]
            pretty_output = f"\nID: {id}\nSummary: {summary}\nScore: {score}\nTitle: {text}\nRating: {rating}\nExplanation: {explanation}\n"
            print(pretty_output)

# Function of input query finds all documents containing the word match
def query_body(input,gte):
    query = {
        "query": {
            "bool": {
                "should": {
                    "multi_match": {
                        "query": input,
                        "type": "most_fields",
                        "fields": ['summary', 'text^3'],
                        "fuzziness": "AUTO",
                    }
                },
                "filter": {
                    "range":{
                        "rating" :{
                            "gte":gte
                        }
                    }
                }
            }
        }
    }

    return query


# Store resuslts of search query & call the function to display results
query = input("Enter your search query: ")
gte = 3
results = es.search(index=index_name ,size=5, body=query_body(query,gte), explain=True)
pretty_response(results)

Enter your search query: best chocolate with rich flavor

ID: B004HLAHUQ
Summary: Best hot chocolate
Score: 59.070366
Title: tried hot chocolate best rich chocolaty apparently chocolate
Rating: 5
Explanation: {'value': 59.070366, 'description': 'sum of:', 'details': [{'value': 3.1643925, 'description': 'weight(summary:best in 87690) [PerFieldSimilarity], result of:', 'details': [{'value': 3.1643925, 'description': 'score(freq=1.0), computed as boost * idf * tf from:', 'details': [{'value': 2.2, 'description': 'boost', 'details': []}, {'value': 2.8189628, 'description': 'idf, computed as log(1 + (N - n + 0.5) / (n + 0.5)) from:', 'details': [{'value': 23478, 'description': 'n, number of documents containing term', 'details': []}, {'value': 393486, 'description': 'N, total number of documents with field', 'details': []}]}, {'value': 0.5102445, 'description': 'tf, computed as freq / (freq + k1 * (1 - b + b * dl / avgdl)) from:', 'details': [{'value': 1.0, 'description': 'freq, occurrences

In [17]:
## EVALUATION

#An input for every hit that we receive, we give it a '1' for relevant and '0' for non relevant
manual_rels, seen = [], set()

for hit in results["hits"]["hits"]:
    docid, text = hit["_source"]["productid"], hit["_source"]["text"]
    if text in seen: continue
    seen.add(text)

    print(f"\nSummary: {hit['_source']['summary']}\nText: {text}\nProduct ID: {docid}")
    rel = input("Relevant? (1 = yes, 0 = no): ")
    while rel not in {'0', '1'}:
        rel = input("Enter 1 or 0: ")
    manual_rels.append((docid, int(rel)))

#Precision @ k
def precision_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = retrieved_docs[:k]
    relevant_count = sum(1 for doc in retrieved_k if doc in relevant_docs)
    return relevant_count / k

#Recall @ k
def recall_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = retrieved_docs[:k]
    relevant_count = sum(1 for doc in retrieved_k if doc in relevant_docs)
    return relevant_count / len(relevant_docs) if relevant_docs else 0

#DCG @ k
def dcg_at_k(scores, k):
    return sum((score / np.log2(idx + 2)) for idx, score in enumerate(scores[:k]))

#NDCG @ k
def ndcg_at_k(retrieved_docs, relevant_docs, k):
    retrieved_scores = [1 if doc in relevant_docs else 0 for doc in retrieved_docs]
    ideal_scores = sorted(retrieved_scores, reverse=True)

    dcg = dcg_at_k(retrieved_scores, k)
    idcg = dcg_at_k(ideal_scores, k)

    return dcg / idcg if idcg > 0 else 0

doc_scores = [(hit["_source"]["productid"], hit["_score"]) for hit in results["hits"]["hits"]]
k = 5

#Manually input retrieved and relevant documents (product IDs) based on personal evaluation
retrieved_docs = ['B004HLAHUQ', 'B000OP5G1E', 'B000OP5G1E', 'B001JDQ4Q6', 'B000H227BG']
relevant_docs = ['B004HLAHUQ', 'B000OP5G1E', 'B000OP5G1E', 'B001JDQ4Q6', 'B000H227BG']

#Get Results
print("Precision@{}: {:.2f}".format(k, precision_at_k(retrieved_docs, relevant_docs, k)))
print("Recall@{}: {:.2f}".format(k, recall_at_k(retrieved_docs, relevant_docs, k)))
print("NDCG@{}: {:.2f}".format(k, ndcg_at_k(retrieved_docs, relevant_docs, k)))


Summary: Best hot chocolate
Text: tried hot chocolate best rich chocolaty apparently chocolate
Product ID: B004HLAHUQ
Relevant? (1 = yes, 0 = no): 1

Summary: best hot chocolate!
Text: hot chocolate rich chocolaty best definitely better commercial mix grocery
Product ID: B000OP5G1E
Relevant? (1 = yes, 0 = no): 1

Summary: This is the best Hot Chocolate
Text: best hot chocolate bought wish discovered time went costa rica brought 9 home vist gift wish brought britt make flavor hot chocolate straight hot chocolate mint hot chocolate line
Product ID: B000OP5G1E
Relevant? (1 = yes, 0 = no): 1

Summary: Best chocolate I ever had
Text: milk chocalate thing best tasted smooth rich best thing soy like chocolate sold usa did eat chocolate 30 year soy chocolate
Product ID: B001JDQ4Q6
Relevant? (1 = yes, 0 = no): 1

Summary: Rich, chocolatey beverage mix
Text: appreciate short list ingredient ghiradelli hot chocolate mix chocolate mocha flavor rich chocolatey overpowered coffee favorite cold weat